# S3 · From REINFORCE to PPO: policy gradients, baselines, advantages and clipping

Study path stage 3 of 7 · Chapter 12 · Day 21 · **DGX Spark GPU first**

A reward model (S1) or a verifier gives **one number per finished answer**. This notebook builds,
line by line, the algorithms that turn that number into an update of the policy's token
probabilities: the exact policy gradient, the REINFORCE estimator, baselines and RLOO, values,
advantages and GAE for credit across tokens, and PPO's KL-shaped rewards, importance ratio,
clipped loss and value loss. At the end you can say exactly what each of PPO's four networks
computes, which ones are trained and on what, and you will have run all four roles on Qwen3-0.6B.

| Part | Question | Discussions |
|---|---|---|
| A · One decision | What is the exact gradient of expected reward, and how does REINFORCE estimate it from samples? | [D23](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md), glossary 13, 17 |
| B · Baselines and RLOO | Why may we subtract a baseline, and which baselines lower the variance? | [D21, D23, D39](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| C · Credit over tokens | How do return, value, Q and advantage split one terminal reward across tokens? What does GAE's λ trade? | [D20, D22–D24, D37–D39](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| D · PPO's pieces | KL-shaped rewards, the clipped ratio loss, the value loss; REINFORCE vs RLOO vs PPO on a toy | [D21, D24–D25, D35–D36](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| E · PPO on Qwen3 | Policy, reference, reward and value in one real PPO loop: what moves, and by how much? | [D24, D25, D35, D36](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |

RL terms (state, action, return, value, advantage, …) are defined at first use below and in the
glossary [RL terms in the LLM setting](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md#-rl-terms-in-the-llm-setting-glossary-d39) of the learning note.

**How the exercises work.** Each exercise has the same rhythm:

1. **Read the task**, then write your lines where you see `...` in the exercise cell.
2. **Run the check.** It tests your code on small cases and prints ✅ or ❌ with a hint. Nothing
   crashes if you haven't finished; you'll see ⏳ instead.
3. **Compare with the reference**, which follows each check and then runs a small experiment.

Later cells use your version if its check passed, otherwise the reference, via
`mine_or_ref(globals(), …)`, so one stuck exercise never blocks the rest. The **Predict** cells ask
for a written answer before you run the reference.

**Data.** Parts A–D use small authored tables (rewards and policies declared in the code).
Part E uses only the *questions* of Anthropic HH-RLHF `helpful-base` (MIT license): train rows from
20,000 on for PPO prompts, the first 64 test questions for evaluation. No preference labels are used.

**Environment.** Kernel `Python (DGX Spark Native)` with CUDA; Qwen3-0.6B and the dataset from the
local Hugging Face cache (nothing is downloaded). Checks and references come from the tested module
`dongxi_llms.study_path.s3_policy_gradient_to_ppo`.

**Run time.** About 4½ minutes on the DGX Spark: Parts A–D take under 30 seconds, the Qwen3 PPO
run about 4 minutes. Peak GPU memory about 23 GiB (two trained 0.6B models with fp32 weights and
AdamW state, one frozen bf16 copy).

**Scale and limits.** Parts A–D are exact: expectations and variances are computed by enumerating
every outcome, so their numbers are facts about the declared toys, not estimates. The toy
comparison in Part D uses five seeds on a 13-state tabular policy; it says nothing about which
algorithm is better for language models. Part E runs PPO for 40 iterations of 16 answers each with
a **rule reward** (finished answer within 48 tokens) in the reward-model role. It shows the four
roles working and measures how far the policy moves on that rule, with standard errors; the rule is
not a quality measure, so no claim about better answers follows.

In [ ]:
import inspect, itertools, math, random, sys, time
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(ROOT / "src"))               # the course package, wherever this copy lives
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from dongxi_llms.study_path.common import mine_or_ref
from dongxi_llms.study_path import qwen as Q
from dongxi_llms.study_path import s3_policy_gradient_to_ppo as s3

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
random.seed(0); np.random.seed(0); torch.manual_seed(0)
BLUE, ORANGE, GRAY, GREEN, RED = "#2563EB", "#EA580C", "#94A3B8", "#16A34A", "#DC2626"
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
torch.set_default_dtype(torch.float64)    # toy parts in float64 so exact identities hold to 1e-12
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

---

# Part A · One decision: the exact gradient and REINFORCE

Start with the smallest case that still has the RL difficulty. One prompt, and the policy picks
one of K = 4 **complete answers** (no tokens yet). The **policy** is a softmax over four logits,
π(a) = softmax(z)_a, and each answer a has a fixed, declared **reward** R_a. The quantity to
maximize is the **expected reward**

  J(z) = Σ_a π(a) R_a.

Here we can compute J and its gradient exactly, because there are only four answers. A language
model can't: its "answers" are all token sequences. So we will also build the estimator used in
practice, which needs only *samples* from π, and test it against the exact gradient.

The fixture: π = [0.4, 0.3, 0.2, 0.1] and R = [1, 2, 3, 6], so J = 0.4 + 0.6 + 0.6 + 0.6 = 2.2.

In [ ]:
z = torch.log(torch.tensor([0.4, 0.3, 0.2, 0.1]))     # logits; softmax(z) = [0.4, 0.3, 0.2, 0.1]
R = torch.tensor([1.0, 2.0, 3.0, 6.0])               # declared reward of each complete answer
p = z.softmax(-1)
J = (p * R).sum()
print("π =", p.tolist(), "| R =", R.tolist(), "| J = Σ π R =", round(float(J), 4))

### Prediction

## Exercise 1 · the exact gradient of expected reward

Differentiate J = Σ_a π(a) R_a with respect to the logits. For a softmax,
∂π(a)/∂z_i = π(a)(1[a = i] − π(i)). Summing over a gives

  ∂J/∂z_i = π(i) R_i − π(i) Σ_a π(a) R_a = π(i) (R_i − J).

Write J and the gradient vector. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_exact_grad(logits, rewards):
    p = logits.softmax(-1)
    J = ...                     # expected reward Σ_a p_a R_a (a scalar)
    return ...                  # ∂J/∂z, shape [K]

In [ ]:
ex1_ok = s3.check_exact_grad(globals().get("my_exact_grad"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def exact_grad_ref(logits, rewards):
    p = logits.softmax(-1)
    return p * (rewards - (p * rewards).sum(-1, keepdim=True))

s3.check_exact_grad(exact_grad_ref, strict=True)
exact_grad = mine_or_ref(globals(), "my_exact_grad", ex1_ok, exact_grad_ref)
grad_J = exact_grad(z, R)
for a in range(4):
    print(f"answer {a}: R = {R[a]:.0f}, R − J = {R[a] - J:+.1f}, ∂J/∂z = {grad_J[a]:+.3f}")

### Reference explanation

### Reference answer and mechanism

∂J/∂z = [−0.48, −0.06, +0.16, +0.38]. Answer 1 earns a **positive** reward (2) but its logit gets a
**negative** gradient, because 2 is below the expected reward 2.2. The exact gradient depends only
on R − J: adding 10 to every reward leaves it unchanged (the check tested this), and the
coordinates sum to zero because adding a constant to all logits doesn't change a softmax.

### REINFORCE: the same gradient from samples

For a language model the sum over answers is out of reach. The **score-function identity**
∇π(a) = π(a) ∇log π(a) turns the exact gradient into an expectation over the policy's own samples:

  ∇J = Σ_a R_a ∇π(a) = Σ_a π(a) R_a ∇log π(a) = E_{a∼π}[ R_a ∇log π(a) ].

So sample N answers a_1 … a_N from π, and average R_{a_i} ∇log π(a_i). This is **REINFORCE**
(Williams, 1992). For a softmax, ∇_z log π(a) = e_a − π (the one-hot vector of a minus the
probabilities), so one sample's estimate is R_a (e_a − π).

In code we never write that vector. We write a **surrogate loss** whose gradient is minus the
estimate, and let autograd do the rest:

  L = −(1/N) Σ_i w_i log π(a_i),  with weights w_i = R_{a_i} treated as data.

The minus sign turns ascent on J into descent on L. The weights must carry no gradient: R is a
number the environment returned, not a function of the logits. The same loss with a sequence
log-probability Σ_t log π(a_t | s_t) is what an LLM policy-gradient step computes.

### Prediction

## Exercise 2 · the REINFORCE surrogate loss

`actions` is a tensor of N sampled answer indices and `weights` has one number per sample (for
now the reward; later a reward minus a baseline). Write the log-probability of each sampled action,
then the scalar loss. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_reinforce_loss(logits, actions, weights):
    logp = ...                  # log π(a_i) for every sampled action, shape [N]
    return ...                  # scalar: −mean of weight × log-prob (weights carry no gradient)

In [ ]:
ex2_ok = s3.check_reinforce_loss(globals().get("my_reinforce_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def reinforce_loss_ref(logits, actions, weights):
    logp = logits.log_softmax(-1)[actions]
    return -(weights.detach() * logp).mean()

s3.check_reinforce_loss(reinforce_loss_ref, strict=True)
reinforce_loss = mine_or_ref(globals(), "my_reinforce_loss", ex2_ok, reinforce_loss_ref)

# Unbiasedness on 20,000 samples: the gradient of the loss is minus the average estimate.
gen = torch.Generator().manual_seed(1)
N = 20_000
actions = torch.multinomial(p, N, replacement=True, generator=gen)
zz = z.clone().requires_grad_(True)
(g_loss,) = torch.autograd.grad(reinforce_loss(zz, actions, R[actions]), zz)
estimate = -g_loss
per_sample = R[actions, None] * s3.score_vectors(p)[actions]          # R_a (e_a − π), [N, K]
se = per_sample.std(0) / N ** 0.5
for k in range(4):
    print(f"coord {k}: REINFORCE mean {estimate[k]:+.4f} ± {se[k]:.4f} (SE) | exact {grad_J[k]:+.4f} | "
          f"|diff| / SE = {abs(estimate[k] - grad_J[k]) / se[k]:.2f}")
print("one-sample spread (std per coordinate):", [round(v, 2) for v in per_sample.std(0).tolist()])

### Reference explanation

### Reference answer and mechanism

The average of 20,000 sampled estimates matches the exact gradient within about two standard
errors in every coordinate: the estimator is **unbiased**. But one sample is very noisy. Its
standard deviation per coordinate (1.0 to 1.7 here) is two to twenty times the size of the
gradient coordinate it estimates (0.06 to 0.48). A single sample of answer 3 (R = 6) pushes *every other* logit down,
including answer 2 whose exact gradient is positive. Part B is about lowering this noise without
changing the mean.

---

# Part B · Baselines and RLOO

Every reward in the fixture is positive, so every sampled answer gets pushed up; the estimator only
learns "which answer is better" through *how much* each is pushed. A **baseline** b is a number
subtracted from the reward before weighting: w = R_a − b. The difference R_a − b is our first
**advantage** estimate: how much better this answer did than expected.

### Prediction

## Predict 1 · shift every reward by +10

Add 10 to every reward (R = [11, 12, 13, 16]) and use plain REINFORCE with one sample.
(a) Does the *expected* gradient change? (b) Does the variance of one sample's estimate change,
and in which direction? (c) What is E_{a∼π}[∇log π(a)], the expected score vector?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
for shift in (0.0, 10.0):
    m = s3.exact_moments(z, R + shift, group=1, advantage_fn=lambda r: r)   # exact: all K outcomes
    print(f"rewards + {shift:4.1f}: exact mean of the estimate {[round(v, 4) for v in m['mean'].tolist()]}"
          f" | variance (trace of the covariance) {m['var']:.3f}")
score_mean = (p[:, None] * s3.score_vectors(p)).sum(0)
print("E[∇log π] =", score_mean.tolist())
assert torch.allclose(score_mean, torch.zeros(4), atol=1e-15)

### Reference explanation

### Reference answer and mechanism

(a) **No.** The mean stays [−0.48, −0.06, 0.16, 0.38]. (c) The expected score vector is **exactly
zero**: E[∇log π] = Σ_a π(a) ∇log π(a) = Σ_a ∇π(a) = ∇ Σ_a π(a) = ∇1 = 0. So for any b that does not
depend on the sampled answer, E[(R − b)∇log π] = ∇J − b·E[∇log π] = ∇J. Shifting all rewards by +10
is the same as using the baseline b = −10. (b) The **variance** grows from 6.2 to 112.6, eighteen
times larger. A baseline never changes the expected gradient; a good one lowers the variance and a
bad one raises it. The usual choice is the expected reward itself, or the **value** V(s) in the
token setting (Part C).

### RLOO: a baseline from the other answers

The best simple baseline, J, is unknown in practice. If we sample **G answers to the same prompt**,
the other G − 1 rewards give a baseline for each answer:

  A_i = R_i − (1/(G−1)) Σ_{j≠i} R_j.

This is **REINFORCE leave-one-out (RLOO)** (Ahmadian et al., 2024). Answer i's baseline uses only
the other answers, which are independent of answer i, so the expected gradient is unchanged.
Subtracting the plain group mean, which includes R_i itself, is *not* the same: it shrinks the
expected gradient by (G − 1)/G (Chapter 12 §12.5).

### Prediction

## Exercise 3 · RLOO advantages

`R` has shape `[..., G]`: rewards of G answers to one prompt in the last axis (there may be a
batch axis in front). Write the leave-one-out baseline. One line; `R.sum(-1, keepdim=True)` is the
group total.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_rloo_advantages(R):
    G = R.shape[-1]
    baseline = ...              # mean of the OTHER G − 1 rewards, same shape as R
    return R - baseline

In [ ]:
ex3_ok = s3.check_rloo(globals().get("my_rloo_advantages"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def rloo_advantages_ref(R):
    G = R.shape[-1]
    return R - (R.sum(-1, keepdim=True) - R) / (G - 1)

s3.check_rloo(rloo_advantages_ref, strict=True)
rloo_advantages = mine_or_ref(globals(), "my_rloo_advantages", ex3_ok, rloo_advantages_ref)
print("R = [0, 1, 3] →", rloo_advantages(torch.tensor([[0.0, 1.0, 3.0]])).tolist())

### Reference explanation

### Reference answer and mechanism

For R = [0, 1, 3]: answer 0's baseline is (1 + 3)/2 = 2, so A = −2; answer 1's is 1.5, so −0.5;
answer 2's is 0.5, so +2.5. If every answer in a group gets the same reward, all advantages are
zero and the group contributes no gradient. That is correct: the group carries no information
about which answer is better.

The next cell compares five estimators that each use **G = 4 answers per update**. For each it
computes the exact mean and variance of the update by enumerating all 4⁴ = 256 ordered groups
(`s3.exact_moments`), so the bars are exact numbers, not Monte Carlo estimates. The left panel
also shows a sampled run: the running average of REINFORCE and RLOO estimates for coordinate 3,
with ±2 standard-error bands.

In [ ]:
G = 4
estimators = {
    "no baseline": lambda r: r,
    "b = J (oracle)": lambda r: r - J,
    "b = J + 5": lambda r: r - J - 5,
    "group mean\n(includes self)": lambda r: r - r.mean(-1, keepdim=True),
    "RLOO": rloo_advantages,
}
moments = {name: s3.exact_moments(z, R, G, fn) for name, fn in estimators.items()}
for name, m in moments.items():
    print(f"{name.replace(chr(10), ' '):28s} bias ‖E[g] − ∇J‖ = {float((m['mean'] - grad_J).norm()):.4f}"
          f" | variance = {m['var']:.4f}")

gen = torch.Generator().manual_seed(2)
n_groups = 3000
groups = torch.multinomial(p.expand(n_groups, 4), G, replacement=True, generator=gen)   # [n_groups, G]
scores = s3.score_vectors(p)
def running(adv_fn):
    g = (adv_fn(R[groups])[..., None] * scores[groups]).mean(1)[:, 3]     # coordinate 3, per group
    n = torch.arange(1, n_groups + 1)
    mean = g.cumsum(0) / n
    sd = torch.sqrt(((g ** 2).cumsum(0) / n - mean ** 2).clamp(min=0))
    return mean, 2 * sd / n.sqrt()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for (name, fn), color in zip([("no baseline", estimators["no baseline"]), ("RLOO", rloo_advantages)], (GRAY, BLUE)):
    mean, band = running(fn)
    x = np.arange(1, n_groups + 1)
    axes[0].plot(x, mean, color=color, label=name)
    axes[0].fill_between(x, mean - band, mean + band, color=color, alpha=0.2)
axes[0].axhline(float(grad_J[3]), color="black", linestyle="--", label=f"exact ∂J/∂z₃ = {grad_J[3]:.2f}")
axes[0].set(xscale="log", xlabel="number of sampled groups (G = 4 answers each)", ylabel="running mean of the estimate",
            title="Both unbiased; RLOO is less noisy", ylim=(-0.5, 1.2))
axes[0].legend(fontsize=8)
names = list(moments)
colors = [GRAY, GREEN, RED, ORANGE, BLUE]
axes[1].bar(range(5), [float((moments[n]["mean"] - grad_J).norm()) for n in names], color=colors)
axes[1].set(xticks=range(5), title="Bias: ‖E[g] − ∇J‖ (exact)")
axes[1].set_xticklabels(names, fontsize=8)
axes[2].bar(range(5), [moments[n]["var"] for n in names], color=colors)
axes[2].set(xticks=range(5), title="Variance of one update (exact)")
axes[2].set_xticklabels(names, fontsize=8)
plt.tight_layout(); plt.show()
assert float((moments["RLOO"]["mean"] - grad_J).norm()) < 1e-12
assert torch.allclose(moments["group mean\n(includes self)"]["mean"], grad_J * (G - 1) / G)

Read the three panels together. All baselines that don't use the sampled answer itself keep the
bias at exactly zero; their variances differ by a factor of ten (oracle J: 0.40, RLOO: 0.56, none:
1.55, a wrong constant J + 5: 4.07). The inclusive group mean has the lowest variance (0.31) but
it is **biased**: its expected update is exactly 3/4 of the true gradient, because each answer is
partly compared with itself. RLOO pays a little variance to remove that bias, and needs no extra
model, only several answers per prompt. GRPO (S4) builds on the inclusive mean, divided by the
group's standard deviation.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Bias and variance of REINFORCE estimators with different baselines (exact), and a sampled running mean](../figures/chapter-12/study-path-s3_policy_gradient_to_ppo-01.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part C · Credit over tokens: return, value, advantage, GAE

A language model writes one **token** at a time, and the reward arrives once, after the last
token. Now the question is not only "was this answer good?" but "which **tokens** made it good?".
This is **credit assignment** ([D38](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md)). The terms, in the LLM setting
([glossary](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md#-rl-terms-in-the-llm-setting-glossary-d39)):

- **state** s_t: the prompt plus the tokens written so far; **action** a_t: the next token;
  the policy gives π(a_t | s_t).
- **reward** r_t: the number received after action t. With a reward model, r_t = 0 for every token
  except the last, which gets R (Part D adds a small KL charge on every token).
- **return** G_t = Σ_k γ^k r_{t+k}: the rewards from t to the end. With γ = 1 and only R at the end,
  G_t = R for every t.
- **value** V(s) = E[G_t | s_t = s]: the expected return if the current policy continues from s.
- **action value** Q(s, a): the expected return if we take token a in s, then follow the policy.
- **advantage** A(s, a) = Q(s, a) − V(s): how much better token a is than the policy's average choice
  in s. The policy-gradient update for token t is proportional to A_t ∇log π(a_t | s_t).

### A toy language with an exact answer

Vocabulary {a, b, c}, answers of exactly T = 3 tokens, so 27 possible answers. The declared
reward rule is

  R(y) = 1[y₂ = b] − 0.5 · 1[y₃ = c]:

the **middle** token decides most of the reward, the **last** token can lose half a point, and the
**first** token doesn't enter the rule. The policy is a table of logits, one row per state
(prefix): 1 + 3 + 9 = 13 states × 3 tokens. At the start the policy rarely writes b in the middle
(π = 0.09) and writes c last with π = 0.50.

In [ ]:
K, T = 3, 3
VOCAB = ["a", "b", "c"]
OFF = [0, 1, 4]                                       # state index of the first prefix of each length
N_STATES = 13
seqs = torch.tensor(list(itertools.product(range(K), repeat=T)))     # all 27 answers, [27, 3]

def toy_reward(y):                                     # y: [N, 3] token ids → R: [N]
    return (y[:, 1] == 1).double() - 0.5 * (y[:, 2] == 2).double()

def toy_states(y):                                     # state index (prefix) before each token, [N, 3]
    return torch.stack([torch.zeros(len(y), dtype=torch.long), 1 + y[:, 0], 4 + 3 * y[:, 0] + y[:, 1]], 1)

def probs_by_depth(theta):                             # [K**d, K] tables of π(a | s) for d = 0, 1, 2
    pr = theta.softmax(-1)
    return [pr[OFF[d]:OFF[d] + K ** d] for d in range(T)]

theta0 = torch.zeros(N_STATES, K)
theta0[1:4] = torch.tensor([0.0, -1.6, 0.0])          # middle token: b is rare
theta0[4:13] = torch.tensor([0.0, 0.0, 0.7])          # last token: c is common
R_all, S_all = toy_reward(seqs), toy_states(seqs)
pi0 = probs_by_depth(theta0)
print("π(· | prompt)      =", [round(v, 3) for v in pi0[0][0].tolist()])
print("π(· | 'a')         =", [round(v, 3) for v in pi0[1][0].tolist()], "(same for every first token)")
print("π(· | 'a b')       =", [round(v, 3) for v in pi0[2][1].tolist()], "(same for every prefix of length 2)")
print("rewards of the 27 answers:", sorted(set(R_all.tolist())))

### Prediction

## Exercise 4 · exact values, action values and advantages

Compute Q and V for every state by working **backwards** from the complete answers. With γ = 1 and
no reward before the end, taking token a in state s leads to the state s·a, so Q(s, a) = V(s·a);
and V(s) = Σ_a π(a | s) Q(s, a).

Indexing: `probs[d]` is `[K**d, K]` (row = prefix of length d in base K). The child of prefix i by
token a has index i·K + a at depth d + 1, so `V_next.view(K**d, K)` puts the children of each prefix
in one row: that *is* Q at depth d. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_exact_values(probs, terminal_reward):
    T, K = len(probs), probs[0].shape[-1]
    Q, V = [None] * T, [None] * T
    V_next = terminal_reward                    # value of a complete answer = its reward, [K**T]
    for d in reversed(range(T)):
        Q[d] = ...                              # [K**d, K]: Q(s, a) = V(s·a)
        V[d] = ...                              # [K**d]:    V(s) = Σ_a π(a|s) Q(s, a)
        V_next = V[d]
    return Q, V

In [ ]:
ex4_ok = s3.check_exact_values(globals().get("my_exact_values"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s3.check_exact_values(s3.ref_exact_values, strict=True)
exact_values = mine_or_ref(globals(), "my_exact_values", ex4_ok, s3.ref_exact_values)
Q0, V0 = exact_values(pi0, R_all)
A0 = [q - v[:, None] for q, v in zip(Q0, V0)]
print(f"V(prompt) = J = {float(V0[0][0]):+.4f}")
print("advantage of each first token  (state = prompt):  ", {VOCAB[a]: round(float(A0[0][0, a]), 3) for a in range(K)})
print("advantage of each middle token (state = 'a'):     ", {VOCAB[a]: round(float(A0[1][0, a]), 3) for a in range(K)})
print("advantage of each last token   (state = 'a b'):   ", {VOCAB[a]: round(float(A0[2][1, a]), 3) for a in range(K)})
y = torch.tensor([[0, 1, 2]])                          # the answer "a b c", R = 0.5
s = toy_states(y)[0]
A_path = [float(A0[d][s[d] - OFF[d], y[0, d]]) for d in range(T)]     # A(s_t, a_t) along the answer
print(f"answer 'a b c' (R = {float(toy_reward(y)):+.1f}): per-token advantages {[round(v, 3) for v in A_path]}"
      f" | REINFORCE (no baseline) gives every token {float(toy_reward(y)):+.1f}")

### Reference explanation

### Reference answer and mechanism

The exact advantages assign credit token by token:

- **first token**: A = 0 for a, b and c. The rule ignores it, and the policy continues the same way
  from every first token, so no first token is better than the average.
- **middle token**: b gets A = +0.91, a and c get −0.09. Picking b is where the reward is earned.
- **last token**: c gets −0.25 and a, b get +0.25 (c is chosen half the time, so the average
  last token costs −0.25).

For the sampled answer "a b c" (R = +0.5) the per-token advantages are [0, +0.91, −0.25]: raise b,
lower c, leave a alone. Plain REINFORCE gives all three tokens the same weight +0.5, pushing up
"a" and even "c". The advantages only average out to the right thing over many samples, which is
the variance Part B measured. A value function lets the update use per-token advantages instead.
In practice V is unknown: PPO **learns** it (the value model) and estimates A from it with GAE.

### Temporal-difference error and GAE

With a value estimate V, the **TD error** of token t compares two consecutive predictions:

  δ_t = r_t + γ V(s_{t+1}) − V(s_t),  with V = 0 after the last token.

If V is exact and transitions are deterministic (appending a token is), δ_t = Q(s_t, a_t) − V(s_t):
the true advantage. If V is wrong, δ_t inherits its errors. The other extreme ignores V after
step t: G_t − V(s_t), the actual outcome minus the prediction. **Generalized advantage estimation**
(GAE; Schulman et al., 2016) mixes the two with a weight λ ∈ [0, 1]:

  A_t = Σ_k (γλ)^k δ_{t+k}  ⇔  A_t = δ_t + γλ A_{t+1}  (computed backwards from the last token).

λ = 0 gives A_t = δ_t; λ = 1 gives A_t = G_t − V(s_t) (the δ's telescope).

**The garlic example** ([D24](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md)): answer tokens "No" · ", raw garlic is" · "toxic" · "." ·
`<end>`, value predictions V = [0.5, 0.1, 0.0, −1.8, −1.9] before each token, final reward R = −2.0
on the last token.

### Prediction

## Exercise 5 · GAE, backwards

Fill in the TD error and the backward recursion. `rewards`, `values`, `mask` are `[B, T]` over
response positions; `mask` is 0 on padding. The given line `next_v` already sets V(s_{t+1}) = 0 after
the last real token. Multiply the running sum by `mask[:, t]` so padding stays 0. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_gae(rewards, values, mask, gamma=1.0, lam=0.95):
    T = rewards.shape[1]
    adv, running = torch.zeros_like(rewards), torch.zeros_like(rewards[:, 0])
    for t in reversed(range(T)):
        next_v = values[:, t + 1] * mask[:, t + 1] if t + 1 < T else torch.zeros_like(running)
        delta = ...                             # δ_t = r_t + γ V(s_{t+1}) − V(s_t)
        running = ...                           # A_t = δ_t + γλ A_{t+1}, zero on padding
        adv[:, t] = running
    return adv

In [ ]:
ex5_ok = s3.check_gae(globals().get("my_gae"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def gae_ref(rewards, values, mask, gamma=1.0, lam=0.95):
    T = rewards.shape[1]
    adv, running = torch.zeros_like(rewards), torch.zeros_like(rewards[:, 0])
    for t in reversed(range(T)):
        next_v = values[:, t + 1] * mask[:, t + 1] if t + 1 < T else torch.zeros_like(running)
        delta = rewards[:, t] + gamma * next_v - values[:, t]
        running = (delta + gamma * lam * running) * mask[:, t]
        adv[:, t] = running
    return adv

s3.check_gae(gae_ref, strict=True)
gae = mine_or_ref(globals(), "my_gae", ex5_ok, gae_ref)

garlic_tokens = s3.GARLIC_TOKENS
V_g = torch.tensor([s3.GARLIC_V]); r_g = torch.tensor([[0, 0, 0, 0, s3.GARLIC_R]]); m_g = torch.ones(1, 5)
lams = [0.0, 0.5, 0.95, 1.0]
A_g = {lam: gae(r_g, V_g, m_g, 1.0, lam)[0] for lam in lams}
print(f"{'token':>18s} " + " ".join(f"λ={lam:<5}" for lam in lams))
for i, tkn in enumerate(garlic_tokens):
    print(f"{tkn!r:>18s} " + " ".join(f"{float(A_g[lam][i]):+7.3f}" for lam in lams))
assert torch.allclose(A_g[0.0], torch.tensor(s3.GARLIC_A_LAM0)) and torch.allclose(A_g[1.0], torch.tensor(s3.GARLIC_A_LAM1))

fig, ax = plt.subplots(figsize=(10, 3.6))
x = np.arange(5); width = 0.16
ax.bar(x - 2 * width, [s3.GARLIC_R] * 5, width, color=GRAY, label="REINFORCE: R on every token")
for j, (lam, color) in enumerate(zip(lams, [RED, ORANGE, GREEN, BLUE])):
    ax.bar(x + (j - 1) * width, A_g[lam].numpy(), width, color=color, label=f"GAE λ = {lam}")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(xticks=x, ylabel="advantage A_t", title="Garlic answer, R = −2.0: where does the blame land?")
ax.set_xticklabels([repr(t) for t in garlic_tokens])
ax.legend(fontsize=8, ncol=3, loc="lower left")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

λ = 0 gives A = [−0.4, −0.1, **−1.8**, −0.1, −0.1]: almost all the blame lands on "toxic", the token
after which V fell from 0.0 to −1.8. λ = 1 gives A = R − V(s_t) = [−2.5, −2.1, −2.0, −0.2, −0.1]:
every early token is blamed, because each is compared with the prediction made *before* it and the
final outcome, with no use of the later predictions. λ = 0.95 is close to λ = 1 here, because the
answer is only five tokens long: (0.95)⁴ ≈ 0.81, so later δ's still count almost fully. REINFORCE
without a baseline (gray) gives every token −2.0.

Which is right? With a correct V, λ = 0 is the true advantage (next predict). With a wrong V, λ = 0
copies its errors into the update.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![GAE advantages of the garlic answer for several λ, compared with REINFORCE's uniform credit](../figures/chapter-12/study-path-s3_policy_gradient_to_ppo-02.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

### Prediction

## Predict 2 · what does λ trade?

Back to the toy language, where everything can be enumerated. The policy-gradient estimate from one
sampled answer is ĝ = Σ_t Â_t ∇log π(a_t | s_t), with Â_t from GAE. Consider three critics:
(i) the **exact** V of the current policy, (ii) V **with errors** (exact V plus fixed noise of
standard deviation 0.3 per state), (iii) **no critic**, V = 0. For each, how do the bias
‖E[ĝ] − ∇J‖ and the variance of ĝ change as λ goes from 0 to 1?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
theta_t = theta0.clone().requires_grad_(True)
J_of = lambda th: (th.log_softmax(-1)[S_all, seqs].sum(1).exp() * R_all).sum()
(grad_true,) = torch.autograd.grad(J_of(theta_t), theta_t)          # exact ∇J, [13, 3]
traj_p = theta0.log_softmax(-1)[S_all, seqs].sum(1).exp()            # π(y) for all 27 answers
pr0 = theta0.softmax(-1)
rew27 = torch.zeros(27, T); rew27[:, -1] = R_all; mask27 = torch.ones(27, T)

V_exact = torch.cat(V0)                                              # one value per state, [13]
noise = torch.randn(N_STATES, generator=torch.Generator().manual_seed(5)) * 0.3
critics = {"exact V": V_exact, "V with errors (σ = 0.3)": V_exact + noise, "no critic (V = 0)": torch.zeros(N_STATES)}

def gradient_moments(V_table, lam):
    A = gae(rew27, V_table[S_all], mask27, 1.0, lam)                  # [27, T]
    g = torch.zeros(27, N_STATES, K)
    for t in range(T):
        score = F.one_hot(seqs[:, t], K).double() - pr0[S_all[:, t]]   # ∇ log π(a_t|s_t) wrt that state's logits
        g[torch.arange(27), S_all[:, t]] += A[:, t:t + 1] * score
    mean = (traj_p[:, None, None] * g).sum(0)
    return float((mean - grad_true).norm()), float((traj_p[:, None, None] * (g - mean) ** 2).sum())

# with the exact V, δ_t IS the true advantage of every token of every answer
A_lam0 = gae(rew27, V_exact[S_all], mask27, 1.0, 0.0)
A_true = torch.stack([A0[d][S_all[:, d] - OFF[d], seqs[:, d]] for d in range(T)], 1)   # exact A(s_t, a_t), [27, 3]
assert torch.allclose(A_lam0, A_true, atol=1e-12)

grid = np.linspace(0, 1, 21)
curves = {name: np.array([gradient_moments(Vt, float(l)) for l in grid]) for name, Vt in critics.items()}
for name, c in curves.items():
    print(f"{name:26s} bias at λ=0 / 0.95 / 1: {c[0, 0]:.4f} / {c[19, 0]:.4f} / {c[-1, 0]:.4f}"
          f" | variance: {c[0, 1]:.3f} / {c[19, 1]:.3f} / {c[-1, 1]:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for (name, c), color in zip(curves.items(), (GREEN, ORANGE, GRAY)):
    axes[0].plot(grid, c[:, 0], color=color, marker=".", label=name)
    axes[1].plot(grid, c[:, 1], color=color, marker=".", label=name)
axes[0].set(xlabel="λ", ylabel="‖E[ĝ] − ∇J‖", title="Bias of the policy gradient (exact)")
axes[1].set(xlabel="λ", ylabel="E‖ĝ − E ĝ‖²", title="Variance of one answer's gradient (exact)")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

- **Exact V:** the bias is 0 at **every** λ. With the true V, E[δ_{t+k} | s_t, a_t] = 0 for k ≥ 1, so
  the extra terms average out; and at λ = 0, Â_t equals the true advantage of every token exactly
  (the `assert` checked all 27 × 3 positions). Variance grows with λ (0.13 at λ = 0 → 0.26 at
  λ = 1), because more of the sampled future enters each token's weight.
- **V with errors:** λ = 0 is **biased** (0.117), because a wrong V(s_{t+1}) depends on the token
  just chosen and shifts that token's update. The bias shrinks to 0 at λ = 1, where only V(s_t),
  which doesn't depend on a_t, remains: a baseline. Its variance is also higher than with the exact
  V at every λ.
- **No critic:** at λ = 0 every δ before the end is 0 and the whole reward lands on the last token.
  That has the lowest variance of all (0.084, because two of three tokens get weight 0) and is
  biased (0.059): the middle token, which earns the reward, gets no credit. At λ = 1, Â_t = R for
  every token: plain REINFORCE, unbiased, variance 0.32.

So λ trades **bias from the critic's errors** against **variance from the sampled future**. A
nearly exact critic allows a small λ; a poor one needs λ near 1. PPO for LLMs commonly uses
λ ≈ 0.95 with γ = 1, and the learned value model is never exact.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Exact bias and variance of the GAE policy gradient against λ, for an exact, a noisy and no critic](../figures/chapter-12/study-path-s3_policy_gradient_to_ppo-03.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part D · PPO's pieces

PPO for language models ([D21–D25, D36](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md)) puts four networks to work. Three produce one number
per token, `[B, T]`; one produces one number per answer, `[B]`:

| Role | Input → output | During PPO | Used for |
|---|---|---|---|
| **policy** π_θ | prefix → next-token distribution; log π_θ(a_t \| s_t) `[B, T]` | trained | sampling, the ratio ρ_t |
| **reference** π_ref | same, a frozen copy of the starting policy; log π_ref `[B, T]` | frozen | the KL charge in the reward |
| **reward model** R | prompt + full answer → one score `[B]` | frozen | the reward on the last token |
| **value model** V_φ | prefix → one number at every position `[B, T]` | trained | the baseline and GAE |

A fifth quantity looks like a network but isn't: π_old, the policy **at sampling time**. It is
stored as log-probabilities `old_logp` `[B, T]` when the batch is sampled ([D25](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md)), and lets
the same batch be reused for several gradient steps.

One PPO iteration: (1) sample answers with π_θ and store `old_logp`; (2) compute log π_ref, R and V
without gradients; (3) build per-token rewards; (4) GAE → advantages and value targets; (5) a few
epochs of the clipped policy loss on π_θ and the value loss on V_φ. You write steps 3 and 5 here.

### Prediction

## Exercise 6 · per-token rewards with a KL charge

PPO maximizes E[R] − β·KL(π_θ ‖ π_ref) ([D35](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md)). The KL is estimated from the sampled tokens
by the log-ratio k1_t = log π(a_t | s_t) − log π_ref(a_t | s_t), and charged **per token** inside the
reward: r_t = −β·k1_t on every response token, and R is added on the last one, at index `last`.
Two lines (the `kl` line is given).

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_kl_shaped_rewards(R, logp, ref_logp, mask, last, beta):
    kl = (logp - ref_logp) * mask               # per-token log-ratio, 0 on padding, [B, T]
    rewards = ...                               # the KL charge on every token, [B, T]
    ...                                         # add R[i] at position last[i] of each row i
    return rewards, kl

In [ ]:
ex6_ok = s3.check_kl_shaped_rewards(globals().get("my_kl_shaped_rewards"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s3.check_kl_shaped_rewards(s3.ref_kl_shaped_rewards, strict=True)
kl_shaped_rewards = mine_or_ref(globals(), "my_kl_shaped_rewards", ex6_ok, s3.ref_kl_shaped_rewards)
# garlic answer, with a policy that has drifted from the reference on "toxic" (log-ratio +0.8)
logp_g = torch.tensor([[-0.5, -1.0, -0.3, -0.1, -0.2]])
ref_g = torch.tensor([[-0.5, -1.1, -1.1, -0.1, -0.2]])
rew_g, kl_g = kl_shaped_rewards(torch.tensor([s3.GARLIC_R]), logp_g, ref_g, torch.ones(1, 5), torch.tensor([4]), 0.05)
for tkn, k_, r_ in zip(s3.GARLIC_TOKENS, kl_g[0].tolist(), rew_g[0].tolist()):
    print(f"{tkn!r:>18s}: log π − log π_ref = {k_:+.2f} → r_t = {r_:+.3f}")
print("sum of the KL charges:", round(float((-0.05 * kl_g).sum()), 3), "| sequence KL estimate:", round(float(kl_g.sum()), 2))

### Reference explanation

### Reference answer and mechanism

Every token pays −β·(log π − log π_ref): here "toxic" pays −0.05 × 0.8 = −0.04 because the policy
has made it more likely than the reference does; ", raw garlic is" pays −0.005. A token the policy
has made *less* likely than the reference gets a small bonus (k1 can be negative on one sample;
its expectation under π is the KL, which is ≥ 0). The last token also receives R = −2.0.

Because the KL charge is inside the reward, it flows through GAE into the advantages like any
other reward: drifting from the reference lowers the advantage of the drifting tokens. That is the
PPO-RLHF convention (InstructGPT). GRPO (S4) instead adds a KL term to the loss.

### The clipped surrogate

PPO reuses each batch for several gradient steps. After the first step, the policy that sampled
the batch (π_old) and the policy being trained (π_θ) differ. The **importance ratio**

  ρ_t = π_θ(a_t | s_t) / π_old(a_t | s_t) = exp(log π_θ − log π_old)

corrects for that: E_{π_old}[ρ A] = E_{π_θ}[A] for a single decision. At ρ = 1 (the first step),
∇ρ_t = ∇log π_θ(a_t | s_t), so ρ_t A_t has exactly the policy-gradient gradient. PPO maximizes the
**clipped surrogate** (Schulman et al., 2017)

  min( ρ_t A_t, clip(ρ_t, 1 − ε, 1 + ε) A_t ),

averaged over the real response tokens. The `min` makes the objective pessimistic: once a token's
probability has moved more than ε in the direction its advantage asks for, the gain stops growing;
a move in the wrong direction is never clipped.

### Prediction

## Exercise 7 · the clipped PPO policy loss

`logp` (current, with gradient), `old_logp` (stored), `adv` and `mask` are `[B, T]`. Write the ratio
and the clipped surrogate; the loss and the clip fraction (share of real tokens with |ρ − 1| > ε, a
standard PPO diagnostic) are given. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_clipped_policy_loss(logp, old_logp, adv, mask, eps=0.2):
    ratio = ...                                 # ρ_t = π_θ / π_old, from log-probs
    surrogate = ...                             # min(ρA, clip(ρ, 1−ε, 1+ε)A), per token
    loss = -(surrogate * mask).sum() / mask.sum()
    clip_frac = (((ratio - 1).abs() > eps).float() * mask).sum() / mask.sum()
    return loss, clip_frac

In [ ]:
ex7_ok = s3.check_clipped_policy_loss(globals().get("my_clipped_policy_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s3.check_clipped_policy_loss(s3.ref_clipped_policy_loss, strict=True)
clipped_policy_loss = mine_or_ref(globals(), "my_clipped_policy_loss", ex7_ok, s3.ref_clipped_policy_loss)

### Reference explanation

### Reference answer and mechanism

`torch.exp(logp - old_logp)` computes the ratio from log-probabilities, which is numerically safe;
never divide probabilities of long sequences. `old_logp` comes from a `no_grad` pass at sampling
time, so it is data; the gradient flows only through `logp`. The checks confirmed the asymmetry:
zero gradient at ρ = 1.5 with A = +1 and at ρ = 0.5 with A = −1 (the helpful moves already beyond
the window), nonzero gradient at ρ = 1.5 with A = −1 and at ρ = 0.5 with A = +1.

### Prediction

## Predict 3 · where is the clipped gradient zero?

With ε = 0.2, sketch the objective min(ρA, clip(ρ, 0.8, 1.2)A) and its slope d/dρ as functions of
ρ ∈ [0.4, 1.8], once for A = +1 and once for A = −1. For which ratios is the slope zero? Is the
slope ever zero inside [0.8, 1.2]?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
rho = torch.linspace(0.4, 1.8, 281)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for a_val, color in ((1.0, BLUE), (-1.0, RED)):
    lp = rho.log().clone().requires_grad_(True)
    adv = torch.full((1, len(rho)), a_val)
    ones = torch.ones(1, len(rho))
    surrogate = -clipped_policy_loss(lp[None], torch.zeros(1, len(rho)), adv, ones, 0.2)[0] * len(rho)
    (g,) = torch.autograd.grad(surrogate, lp)
    slope = g / rho                                   # d/dρ = (d/d log ρ) / ρ
    obj = torch.min(rho * a_val, rho.clamp(0.8, 1.2) * a_val)
    axes[0].plot(rho, obj, color=color, label=f"A = {a_val:+.0f}")
    axes[0].plot(rho, rho * a_val, color=color, linestyle=":", alpha=0.5)
    axes[1].plot(rho, slope.detach(), color=color, label=f"A = {a_val:+.0f}")
    zero = rho[slope.abs() < 1e-9]
    print(f"A = {a_val:+.0f}: slope 0 for ρ in [{zero.min():.2f}, {zero.max():.2f}]" if len(zero) else "no zero slope")
for ax in axes:
    ax.axvspan(0.8, 1.2, color=GREEN, alpha=0.08)
    ax.set_xlabel("ratio ρ = π_θ / π_old")
axes[0].set(ylabel="surrogate (maximized)", title="Clipped objective (dotted: unclipped ρA)")
axes[1].set(ylabel="d surrogate / dρ", title="Slope, by autograd through your loss")
axes[0].legend(); plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

- **A = +1:** the slope is +1 up to ρ = 1.2 and **0 above it**: once the good token is 20% more
  likely than at sampling time, this batch stops pushing it. Below 0.8 the slope is still +1: a
  good token that became *less* likely is pushed back up without limit.
- **A = −1:** the slope is −1 down to ρ = 0.8 and **0 below it**. Above 1.2 the slope is still −1:
  a bad token that became *more* likely is always pulled back.
- Inside [0.8, 1.2] the slope is never zero: there PPO is the ordinary policy gradient.

So clipping removes the incentive only for moves that already went far enough in the helpful
direction. It doesn't bound the KL or the ratio itself: other tokens share the same weights and can
move, and the ratio can leave the window during the epochs (Chapter 12 §12.7).

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![PPO clipped surrogate and its slope against the probability ratio, for positive and negative advantage](../figures/chapter-12/study-path-s3_policy_gradient_to_ppo-04.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

### Prediction

## Exercise 8 · the value loss and its targets

The value model is trained by regression. Its target at token t is the **λ-return**
G^λ_t = A_t + V_old(s_t): the GAE advantage plus the value predicted at sampling time. With λ = 1
this is the Monte-Carlo return G_t; with λ = 0 it is the TD target r_t + γV(s_{t+1}). The targets
are data. Write the target and the masked mean-squared error. Two lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_value_loss(new_values, old_values, adv, mask):
    returns = ...                               # λ-return targets, no gradient
    return ...                                  # masked mean of (V_new − target)²

In [ ]:
ex8_ok = s3.check_value_loss(globals().get("my_value_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s3.check_value_loss(s3.ref_value_loss, strict=True)
value_loss = mine_or_ref(globals(), "my_value_loss", ex8_ok, s3.ref_value_loss)
for lam in (0.0, 1.0):
    a = gae(r_g, V_g, m_g, 1.0, lam)
    print(f"garlic, λ = {lam}: value targets {[round(v, 2) for v in (a + V_g)[0].tolist()]}")

### Reference explanation

### Reference answer and mechanism

With λ = 1 every garlic prefix gets the target −2.0, the reward this answer actually earned: the
value model learns that, under the current policy, these prefixes tend to end at −2. With λ = 0
the targets are the next prediction, [0.1, 0.0, −1.8, −1.9, −2.0], so V is pulled toward its own
next value (bootstrapping). In both cases only the answer-level R enters, yet every prefix gets a
target: that is how a value model learns per-token predictions from one score per answer ([D37](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md)).
The targets change every iteration, because V(s) is the expected return **of the current policy**
([D25](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md)).

### Assemble: REINFORCE, RLOO and PPO on the toy language

The next cell trains the toy policy (13 × 3 logits, start θ₀ from Part C) with four methods, each
for 100 updates of **8 sampled answers**, with the same learning rate and 5 seeds. It uses the
functions you wrote (or the references):

- **REINFORCE:** weight R on every token of an answer.
- **RLOO:** weight `rloo_advantages(R)` on every token.
- **PPO, 1 epoch** and **PPO, 4 epochs:** a learned value table V (13 numbers, starting at 0, trained
  with `value_loss`), `kl_shaped_rewards` with β = 0.02 against the frozen θ₀, `gae` with λ = 0.95,
  and `clipped_policy_loss` with ε = 0.2. The four roles are all present: policy table, frozen
  reference table, the reward rule, the value table.

The metric is the **exact** expected reward J(θ) (sum over all 27 answers), and the exact sequence
KL(π_θ ‖ π_θ₀). As a sanity check, PPO with one epoch, β = 0, λ = 1 and V fixed at 0 must reproduce
REINFORCE exactly: then ρ = 1, the clip is inactive, and A_t = R for every token.

In [ ]:
def toy_logp(theta, y, S):
    return theta.log_softmax(-1)[S, y]                                   # [N, T]

def exact_J(theta):
    return float((toy_logp(theta, seqs, S_all).sum(1).exp() * R_all).sum())

def exact_kl(theta):
    lp, lr = toy_logp(theta, seqs, S_all).sum(1), toy_logp(theta0, seqs, S_all).sum(1)
    return float((lp.exp() * (lp - lr)).sum())

def toy_sample(theta, n, gen):
    y = torch.zeros(n, T, dtype=torch.long)
    for t in range(T):
        states = toy_states(y)[:, t]                                    # prefix before token t
        y[:, t] = torch.multinomial(theta.softmax(-1)[states], 1, generator=gen).squeeze(1)
    return y

def train_toy(method, seed, updates=100, n=8, lr=1.0, epochs=4, beta=0.02, lam=0.95, value_lr=1.0, learn_value=True):
    gen = torch.Generator().manual_seed(seed)
    theta = theta0.clone().requires_grad_(True)
    V_table = torch.zeros(N_STATES)
    J_hist, kl_hist, passes = [exact_J(theta0)], [0.0], [0]
    for _ in range(updates):
        y = toy_sample(theta.detach(), n, gen); S = toy_states(y); Rs = toy_reward(y)
        mask = torch.ones(n, T)
        old = toy_logp(theta.detach(), y, S)
        if method in ("REINFORCE", "RLOO"):
            w = Rs if method == "REINFORCE" else rloo_advantages(Rs)
            loss = -(w[:, None] * toy_logp(theta, y, S)).sum(1).mean()      # sequence log-prob × weight
            (g,) = torch.autograd.grad(loss, theta)
            with torch.no_grad():
                theta -= lr * g
            passes.append(passes[-1] + 1)
        else:                                                              # PPO
            rewards, _ = kl_shaped_rewards(Rs, old, toy_logp(theta0, y, S), mask, torch.full((n,), T - 1), beta)
            V_old = V_table[S]
            adv = gae(rewards, V_old, mask, 1.0, lam)
            for _e in range(epochs):
                loss, _ = clipped_policy_loss(toy_logp(theta, y, S), old, adv, mask, 0.2)
                (g,) = torch.autograd.grad(loss, theta)
                with torch.no_grad():
                    theta -= lr * T * g          # × T: the loss averages over tokens, REINFORCE sums them
                if learn_value:
                    Vt = V_table.clone().requires_grad_(True)
                    (gv,) = torch.autograd.grad(value_loss(Vt[S], V_old, adv, mask), Vt)
                    V_table = V_table - value_lr * gv
            passes.append(passes[-1] + epochs)
        J_hist.append(exact_J(theta.detach())); kl_hist.append(exact_kl(theta.detach()))
    return {"J": np.array(J_hist), "kl": np.array(kl_hist), "passes": np.array(passes), "V": V_table}

# sanity check: PPO with 1 epoch, β = 0, λ = 1 and V ≡ 0 is REINFORCE
a = train_toy("REINFORCE", 0, updates=30)
b = train_toy("PPO", 0, updates=30, epochs=1, beta=0.0, lam=1.0, learn_value=False)
assert np.allclose(a["J"], b["J"], atol=1e-10), "PPO(1 epoch, no KL, no critic) should equal REINFORCE"
print("PPO with 1 epoch, β = 0, λ = 1, V = 0 reproduces REINFORCE exactly over 30 updates ✓")

t0 = time.time()
SEEDS = range(5)
arms = {"REINFORCE": dict(method="REINFORCE"), "RLOO": dict(method="RLOO"),
        "PPO, 1 epoch": dict(method="PPO", epochs=1), "PPO, 4 epochs": dict(method="PPO", epochs=4)}
toy_runs = {name: [train_toy(seed=s, **kw) for s in SEEDS] for name, kw in arms.items()}
print(f"trained {len(arms)} methods × {len(SEEDS)} seeds in {time.time() - t0:.1f}s; start J = {exact_J(theta0):+.3f}, best possible J = 1")
for name, runs in toy_runs.items():
    at = lambda k: np.array([r["J"][k] for r in runs])
    print(f"{name:14s} J after 10 / 25 / 100 updates: " + " / ".join(f"{at(k).mean():.3f} ± {at(k).std(ddof=1) / len(runs) ** 0.5:.3f}" for k in (10, 25, 100))
          + f" | KL at 100: {np.mean([r['kl'][-1] for r in runs]):.2f} | gradient passes: {runs[0]['passes'][-1]}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for (name, runs), color in zip(toy_runs.items(), (GRAY, BLUE, ORANGE, RED)):
    Js = np.stack([r["J"] for r in runs]); kls = np.stack([r["kl"] for r in runs])
    answers = np.arange(Js.shape[1]) * 8
    mean, se = Js.mean(0), Js.std(0, ddof=1) / np.sqrt(len(runs))
    axes[0].plot(answers, mean, color=color, label=name)
    axes[0].fill_between(answers, mean - 2 * se, mean + 2 * se, color=color, alpha=0.15)
    axes[1].plot(runs[0]["passes"], mean, color=color, label=name)
    axes[2].plot(answers, kls.mean(0), color=color, label=name)
axes[0].set(xlabel="sampled answers", ylabel="exact expected reward J", title="Same samples (±2 SE over 5 seeds)")
axes[1].set(xlabel="gradient passes", ylabel="exact expected reward J", title="Same optimizer work", xscale="log")
axes[2].set(xlabel="sampled answers", ylabel="exact KL(π ‖ π_ref)  [nats]", title="Distance from the start policy")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

Read the three panels together (numbers printed above):

- **Per sampled answer**, PPO with 4 epochs learns fastest by a wide margin: it reaches J ≈ 0.9
  after 25 updates (200 answers), where the others are near 0.3. But it also does **4× the gradient
  passes** (middle panel). PPO with 1 epoch, which has the critic, the KL and the clip but no reuse,
  is not faster than REINFORCE or RLOO here. So in this toy the gain comes from **reusing each batch
  several times**, which the clipping makes safe enough, not from the critic.
- **RLOO vs REINFORCE**: about the same here. The toy's rewards are already close to centered
  (between −0.5 and 1), so a baseline removes less variance than in Part A.
- **KL** keeps growing for all methods because the best policy here is deterministic (b, then not
  c). β = 0.02 is too small to stop that; the KL-regularized optimum is π_ref · exp(R/β) normalized
  (S2), which for β = 0.02 is still almost deterministic.

The seeds and learning rate are fixed, not tuned per method; the ranking can change with other
learning rates, batch sizes or tasks. This is an exact toy with 13 states: no claim about LLM-scale
algorithm choice follows from it.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![REINFORCE, RLOO and PPO on the toy token task: expected reward against samples and gradient passes, and KL](../figures/chapter-12/study-path-s3_policy_gradient_to_ppo-05.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part E · PPO on Qwen3-0.6B: all four roles

Now the same loop on a real model. The four roles:

| Role | In this run | Trained? | Output per batch |
|---|---|---|---|
| **policy** π_θ | Qwen3-0.6B chat model, fp32 weights | yes, AdamW, lr 10⁻⁶ | log π_θ `[B, T]` |
| **reference** π_ref | a second copy of Qwen3-0.6B, bf16 | frozen | log π_ref `[B, T]` |
| **reward model** | a **rule function** (no network, see below) | — | R `[B]` |
| **value model** V_φ | Qwen3-0.6B body + a new `Linear(1024 → 1)` head, zero-initialized, fp32 | yes (all but the token embedding), AdamW, lr 10⁻⁵ | V(s_t) `[B, T]` |
| π_old | log π_θ stored at sampling time | — | `old_logp` `[B, T]` |

**Why a rule reward.** A reward model trained on preference pairs (S1) is a noisy proxy, and a
few dozen PPO iterations don't move its score beyond its noise (Day 15 · 04 Part E measured a change
within one standard error). To see PPO change the policy measurably in five minutes we use a
reward we can compute exactly and that the base policy often fails:

  R = 1.0 if the answer **ends with `<|im_end|>` within 48 tokens**; 0.5 if it ends within the
  96-token sampling limit but is longer; 0.0 if it is cut off at 96 tokens.

Qwen3-0.6B (thinking off) writes long answers to HH-RLHF questions: about three quarters are cut
off at 96 tokens. The rule fills the **reward-model role** exactly as a reward model would: it reads
the finished sample and returns one number, which is placed on the last token. It measures length
and termination, not helpfulness: a policy can satisfy it by writing short, unhelpful answers.

**Precision.** The two trained networks keep their weights in **float32** and run their forward
passes in bf16 under `torch.autocast`. Training bf16 weights directly would silently drop most
updates: AdamW moves each weight by about the learning rate (10⁻⁶ here), while bf16 has only 8
significant bits, so a weight of 0.02 can only change in steps of about 1.2·10⁻⁴. The next cell
shows this on a real weight matrix. The frozen reference stays in bf16 (it is never updated).
Because the fp32 policy and the bf16 reference compute slightly different log-probabilities, the
measured KL is about 0.1 nats per answer before any training, a numerical floor, not drift.

The next cell loads the models and the questions and prints the shapes each role produces. The
sampling and log-probability helpers come from `dongxi_llms.study_path.qwen`; their source is shown.

In [ ]:
torch.set_default_dtype(torch.float32)
bf16 = lambda: torch.autocast("cuda", dtype=torch.bfloat16)        # bf16 compute, fp32 master weights
tok = Q.load_tokenizer()
policy = Q.load_chat_model(dtype=torch.float32)                    # trained
ref = Q.load_chat_model().eval().requires_grad_(False)             # frozen copy of the starting policy (bf16)
value = Q.load_reward_model(tok, dtype=torch.float32)              # Qwen3 body + Linear(1024 → 1) head, trained
torch.nn.init.zeros_(value.score.weight)                           # start at V = 0 everywhere
value.model.embed_tokens.requires_grad_(False)                     # saves ~2 GiB of optimizer state

w = ref.model.layers[10].mlp.up_proj.weight.data                   # a real bf16 weight matrix
print(f"bf16 trap: adding 1e-6 leaves {((w + 1e-6) == w).float().mean():.1%} of this matrix unchanged in bf16, "
      f"{((w.float() + 1e-6) == w.float()).float().mean():.1%} in fp32")
train_questions = Q.hh_questions("train", start=20000)
eval_questions = Q.hh_questions("test")[:64]
print(f"{len(train_questions)} training questions, {len(eval_questions)} evaluation questions")
print(inspect.getsource(Q.rollout))
print(inspect.getsource(Q.token_logps))

def values_of(model, ro):
    """V(s_t) for every response position: the scalar head read at the token BEFORE a_t."""
    h = model.model(input_ids=ro["ids"], attention_mask=ro["attn"], position_ids=ro["pos"]).last_hidden_state
    return model.score(h).squeeze(-1).float()[:, ro["P"] - 1:-1] * ro["mask"]

torch.manual_seed(0)
with bf16():
    ro = Q.rollout(policy, tok, eval_questions[:4], max_new=96)
with torch.no_grad(), bf16():
    shapes = {"policy log π": Q.token_logps(policy, ro).shape, "reference log π_ref": Q.token_logps(ref, ro).shape,
              "value V(s_t)": values_of(value, ro).shape}
for name, shp in shapes.items():
    print(f"{name:22s} {tuple(shp)}")
print("response lengths:", ro["mask"].sum(1).int().tolist(), "| cut off at 96:", ro["truncated"].tolist())

### Prediction

## Exercise 9 · the rule reward

`lengths` is `[B]` (response tokens including `<|im_end|>`), `finished` is `[B]` bool (the answer
ended with `<|im_end|>` before the limit). Return 1.0 for finished answers with `lengths <= target`,
0.5 for finished answers that are longer, 0.0 for answers that were cut off. One line with
`torch.where`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_rule_reward(lengths, finished, target=48):
    return ...                                  # [B] float: 1.0 / 0.5 / 0.0

In [ ]:
ex9_ok = s3.check_rule_reward(globals().get("my_rule_reward"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s3.check_rule_reward(s3.ref_rule_reward, strict=True)
rule_reward = mine_or_ref(globals(), "my_rule_reward", ex9_ok, s3.ref_rule_reward)
TARGET, MAX_NEW = 48, 96

def score_rollout_rule(ro):
    """The reward-model role: one number per finished sample."""
    return rule_reward(ro["mask"].sum(1), ~ro["truncated"], TARGET).float().to(ro["mask"].device)

print("rewards of the 4 samples above:", score_rollout_rule(ro).tolist())

### Reference explanation

### Reference answer and mechanism

`torch.where(cond, x, y)` picks x where cond is true. The truncation test comes first: a cut-off
answer earns 0 however short its visible part is. The reward is computed from the rollout alone
(token count and the `<|im_end|>` flag), so it is deterministic and costs nothing, unlike a reward
model's forward pass. It plays the same role: one number per answer, placed on the last token.

### Prediction

## Exercise 10 · the PPO signals of one batch

Inside one PPO iteration, after sampling, you have `R` `[B]`, `old_logp`, `ref_logp`, `V` (all
`[B, T]`) and the rollout dict `ro` with `mask` and `last`. Combine your earlier functions:

1. per-token rewards and KL with `kl_shaped_rewards` (Exercise 6);
2. advantages with `gae` (Exercise 5), γ = 1;
3. value targets: the λ-return, `adv + V`, **before** normalizing;
4. **whiten** the advantages: subtract their mean and divide by their standard deviation over the
   real response tokens `adv[m]` of the batch, then zero the padding. This keeps the policy step size
   independent of the reward's scale; it is standard in PPO implementations.

Four lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
BETA, LAM = 0.05, 0.95

def my_ppo_signals(R, old_logp, ref_logp, V, ro, beta=BETA, lam=LAM):
    mask, m = ro["mask"], ro["mask"].bool()
    rewards, kl = ...                           # step 1
    adv = ...                                   # step 2
    returns = ...                               # step 3
    adv = ...                                   # step 4
    return rewards, adv, returns, kl

In [ ]:
BETA, LAM = 0.05, 0.95
ex10_ok = s3.check_ppo_signals(globals().get("my_ppo_signals"), beta=BETA, lam=LAM)

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def ppo_signals_ref(R, old_logp, ref_logp, V, ro, beta=BETA, lam=LAM):
    return s3.ref_ppo_signals(R, old_logp, ref_logp, V, ro["mask"], ro["last"], beta, lam,
                              kl_fn=kl_shaped_rewards, gae_fn=gae)

s3.check_ppo_signals(ppo_signals_ref, beta=BETA, lam=LAM, strict=True)
ppo_signals = mine_or_ref(globals(), "my_ppo_signals", ex10_ok, ppo_signals_ref)

### Reference explanation

### Reference answer and mechanism

The order matters in one place: the value targets use the **raw** GAE advantages (`returns = adv +
V` is the λ-return in reward units). Whitening first would give the value model targets in
standard-deviation units that change from batch to batch. The whitening uses only real tokens
(`adv[m]`), so padding doesn't pull the mean toward 0. After whitening, about half the tokens of
the batch have positive advantage and half negative, whatever the reward's offset: a batch-level
baseline on top of V.

### The PPO loop

Everything below is given; it calls your `ppo_signals`, `clipped_policy_loss` and `value_loss`.
Each iteration samples 16 answers (one per question) at temperature 1 with at most 96 new tokens,
computes the signals without gradients, then runs **2 epochs** over the same batch: one AdamW step
on the policy with the clipped loss, one on the value model with the value loss. Gradients are
clipped to norm 1. The fixed evaluation samples one answer for each of 64 held-out test questions
with a fixed seed, before and after training.

In [ ]:
N_ITERS, BATCH, EPOCHS, EPS = 40, 16, 2, 0.2
policy_opt = torch.optim.AdamW(policy.parameters(), lr=1e-6, foreach=False)
value_params = [q for q in value.parameters() if q.requires_grad]
value_opt = torch.optim.AdamW(value_params, lr=1e-5, foreach=False)

def ppo_iteration(questions):
    with bf16():
        ro = Q.rollout(policy, tok, questions, max_new=MAX_NEW)                        # 1 · sample, π_old
    with torch.no_grad(), bf16():
        old_logp, ref_logp, V = Q.token_logps(policy, ro), Q.token_logps(ref, ro), values_of(value, ro)
    R = score_rollout_rule(ro)                                                        # 2 · reward role
    rewards, adv, returns, kl = ppo_signals(R, old_logp, ref_logp, V, ro)             # 3–4 · rewards, GAE
    for _ in range(EPOCHS):                                                           # 5 · two updates
        with bf16():
            logp = Q.token_logps(policy, ro)
        p_loss, clip_frac = clipped_policy_loss(logp, old_logp, adv, ro["mask"], EPS)
        p_loss.backward(); torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        policy_opt.step(); policy_opt.zero_grad()
        with bf16():
            new_v = values_of(value, ro)
        v_loss = value_loss(new_v, V, returns - V, ro["mask"])
        v_loss.backward(); torch.nn.utils.clip_grad_norm_(value_params, 1.0)
        value_opt.step(); value_opt.zero_grad()
    return {"reward": R.mean().item(), "kl": kl.sum(1).mean().item(), "value_loss": v_loss.item(),
            "clip_frac": clip_frac.item(), "length": ro["mask"].sum(1).mean().item(),
            "finished": (~ro["truncated"]).float().mean().item()}

@torch.no_grad()
def evaluate(model, seed=123):
    """One sample per evaluation question (fixed seed): rule reward, length, finished flag, sequence KL."""
    torch.manual_seed(seed)
    out = {"reward": [], "length": [], "finished": [], "kl": [], "texts": []}
    for i in range(0, len(eval_questions), 32):
        with bf16():
            ro_e = Q.rollout(model, tok, eval_questions[i:i + 32], max_new=MAX_NEW)
            kl_e = ((Q.token_logps(model, ro_e) - Q.token_logps(ref, ro_e)) * ro_e["mask"]).sum(1)
        out["reward"].append(score_rollout_rule(ro_e).cpu())
        out["length"].append(ro_e["mask"].sum(1).cpu())
        out["finished"].append((~ro_e["truncated"]).float().cpu())
        out["kl"].append(kl_e.float().cpu())
        out["texts"] += Q.response_texts(tok, ro_e)
    return {k: (v if k == "texts" else torch.cat(v)) for k, v in out.items()}

@torch.no_grad()
def fraction_changed(trained, start):
    """Share of weights that differ from the starting checkpoint, compared at bf16 resolution."""
    start_params = dict(start.named_parameters())
    n = changed = 0
    for name, q in trained.named_parameters():
        changed += (q.to(torch.bfloat16) != start_params[name]).sum().item(); n += q.numel()
    return changed / n

ref_fingerprint = [float(q.float().sum()) for q in list(ref.parameters())[:3]]
print(f"before training: policy weights changed {fraction_changed(policy, ref):.1%}, "
      f"value body changed {fraction_changed(value.model, ref.model):.1%}")
t0 = time.time()
eval_before = evaluate(policy)
print(f"evaluation before training ({time.time() - t0:.0f}s): rule reward {eval_before['reward'].mean():.3f}, "
      f"length {eval_before['length'].mean():.1f}, finished {eval_before['finished'].mean():.2f}")

### Prediction

## Predict 4 · what will PPO do to KL and length?

The loop will run 40 iterations × 16 answers = 640 sampled answers. Predict, before running it:
(a) the direction and rough size of the change in the evaluation rule reward and the mean length;
(b) how the KL from the reference evolves (it starts at 0, since policy = reference);
(c) what the value loss and the clip fraction do over training.

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
torch.manual_seed(1); random.seed(1)
torch.cuda.reset_peak_memory_stats()
t0 = time.time(); ppo_log = []
for it in range(N_ITERS):
    ppo_log.append(ppo_iteration(random.sample(train_questions, BATCH)))
    if it % 10 == 9:
        recent = {k: np.mean([row[k] for row in ppo_log[-10:]]) for k in ppo_log[0]}
        print(f"iter {it + 1:3d} | " + " | ".join(f"{k} {v:.3f}" for k, v in recent.items()) + f" | {time.time() - t0:.0f}s")
train_seconds = time.time() - t0
peak_gib = Q.gib(torch.cuda.max_memory_allocated())
eval_after = evaluate(policy)

def mean_se(x):
    return f"{x.float().mean():.3f} ± {x.float().std() / len(x) ** 0.5:.3f}"
print(f"\ntraining {train_seconds:.0f}s | peak GPU memory {peak_gib:.1f} GiB")
print(f"{'evaluation (64 questions)':28s} {'before':>16s} {'after':>16s} {'paired change':>18s}")
for key in ("reward", "length", "finished", "kl"):
    b, a = eval_before[key], eval_after[key]
    print(f"{key:28s} {mean_se(b):>16s} {mean_se(a):>16s} {mean_se(a - b):>18s}")
assert all(np.isfinite([row[k] for row in ppo_log for k in row]))

### Reference explanation

### Reference answer and mechanism

Compare your prediction with the printed table. In the reference run (seeded; your numbers can
differ) the evaluation rule reward rose from 0.12 to 0.50 (paired change +0.38 ± 0.05 SE), the
mean length fell from 90.3 to 62.9 tokens (−27 ± 3), the finished share rose from 0.17 to 0.59, and
the sequence KL from the reference rose from 0.1 (the fp32/bf16 floor) to 6.4 ± 0.8 nats. The
mechanisms:

- **Reward and length.** Every finished answer within 48 tokens gets the largest advantage, so the
  policy raises the probability of `<|im_end|>` earlier in the answer. Length falls and the rule
  reward rises. The paired change on the 64 fixed questions, with its standard error, says whether
  the change is beyond sampling noise.
- **KL** starts near 0 and grows faster and faster (batch log: about 0.2 per answer over
  iterations 1–10, 3.7 over iterations 31–40): the policy moves away from the reference in the
  tokens that end answers early. The KL charge (β = 0.05) costs 0.05 × 6.4 ≈ 0.32 per answer at the
  end, less than the +0.38 the rule now pays, so it slows the drift but doesn't stop it.
- **Value loss** stays small (below 0.01) and shows no clear trend: most rewards are 0, so the
  targets are small, and the value model chases a moving target as the policy's expected reward
  rises. A falling value loss is not guaranteed in PPO, because the policy changes the targets.
- **Clip fraction** stays below about 1%: with lr 10⁻⁶ and 2 epochs per batch, few token ratios
  leave [0.8, 1.2] before the batch is discarded. Almost all of the update happens inside the clip
  window; clipping matters more with more epochs or larger steps.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.3))
panels = [("reward", "rule reward (batch mean)", BLUE), ("kl", "KL from ref, per answer", ORANGE),
          ("value_loss", "value loss", GREEN), ("clip_frac", "clip fraction", RED),
          ("length", "answer length (tokens)", GRAY)]
for ax, (key, title, color) in zip(axes, panels):
    series = np.array([row[key] for row in ppo_log])
    ax.plot(np.arange(1, len(series) + 1), series, color=color, alpha=0.35)
    smooth = np.convolve(series, np.ones(5) / 5, mode="valid")
    ax.plot(np.arange(5, len(series) + 1), smooth, color=color)
    ax.set(title=title, xlabel="PPO iteration")
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![PPO on Qwen3-0.6B with the rule reward: reward, KL, value loss, clip fraction and length per iteration](../figures/chapter-12/study-path-s3_policy_gradient_to_ppo-06.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

### Which networks changed, and what did the answers become?

The four roles predict which weights move: policy and value are trained, the reference is frozen,
and the rule has no weights. The next cell checks that on the actual weights, and prints the
evaluation answers to the first three questions before and after training. A higher rule reward
only says the answers end sooner; read them to judge what was lost or kept. In the reference run one
"after" answer contains a Chinese phrase in the middle of an English sentence. That is a typical
sign of a policy that has drifted far from its reference (KL ≈ 6.4 nats per answer) on a reward
that never checks language or content. S7 studies such failures.

In [ ]:
frac_policy, frac_value = fraction_changed(policy, ref), fraction_changed(value.model, ref.model)
print(f"policy: {frac_policy:.1%} of weights differ from the start (at bf16 resolution)")
print(f"value:  {frac_value:.1%} of body weights differ; head |w| sum = {value.score.weight.abs().sum():.3f} (started at 0)")
print("ref:    unchanged" if [float(q.float().sum()) for q in list(ref.parameters())[:3]] == ref_fingerprint else "ref: CHANGED")
assert [float(q.float().sum()) for q in list(ref.parameters())[:3]] == ref_fingerprint
assert frac_policy > 0.01 and frac_value > 0.01 and value.score.weight.abs().sum() > 0
for i in range(3):
    print(f"\nQ: {eval_questions[i]}")
    for tag, ev in (("before", eval_before), ("after ", eval_after)):
        txt = ev["texts"][i].replace("\n", " ")
        print(f"  {tag} [{int(ev['length'][i])} tok, R = {ev['reward'][i]:.1f}]: {txt[:240]}")

---

# Summary

| What you built | Where | What was measured |
|---|---|---|
| exact gradient ∂J/∂z = π(R − J) | Ex 1 | a rewarded answer can get a negative gradient |
| REINFORCE surrogate loss | Ex 2 | unbiased (20,000 samples within ~2 SE), one-sample spread several times the gradient |
| RLOO advantages | Ex 3 | exact bias 0; variance 0.56 vs 1.55 without a baseline; the inclusive mean is biased by (G−1)/G |
| exact V, Q, A of a token policy | Ex 4 | first token A = 0, decisive middle token A = +0.91 |
| GAE, backwards | Ex 5 | garlic: λ = 0 → [−0.4, −0.1, −1.8, −0.1, −0.1]; λ = 1 → [−2.5, −2.1, −2.0, −0.2, −0.1] |
| — | Predict 2 | λ trades critic bias against sampling variance (exact) |
| KL-shaped per-token rewards | Ex 6 | −β·(log π − log π_ref) on every token, R on the last |
| clipped surrogate | Ex 7 + Predict 3 | zero slope only beyond the window in the helpful direction |
| value loss with λ-return targets | Ex 8 | λ = 1 → Monte-Carlo return, λ = 0 → TD target |
| toy comparison | Part D | PPO's speed per sample comes from reusing batches (4 epochs); 1-epoch PPO ≈ REINFORCE ≈ RLOO here |
| rule reward, PPO signals | Ex 9–10, Part E | four roles on Qwen3-0.6B; rule reward 0.12 → 0.50 (+0.38 ± 0.05), length −27 ± 3 tokens, KL 6.4 ± 0.8 nats; 11% of policy weights moved |

**Evidence.** Parts A–D are exact computations on declared toys: their numbers are not estimates,
and the `assert`s hold to 10⁻¹⁰ or better. The toy comparison is seeded (5 seeds, ±2 SE bands) and
uses one learning rate for all methods; it shows mechanisms (batch reuse, the role of the critic,
KL growth), not a ranking of algorithms for language models. Part E verifies the mechanics of PPO
with all four roles on a real 0.6B model and measures, on 64 fixed held-out questions with paired
standard errors, how the policy moved under a rule reward: in the reference run +0.38 ± 0.05 in
rule reward and −27 ± 3 tokens in length, both far beyond their standard errors, at a KL of
6.4 nats. The trained weights are fp32, and the run checks that they really moved (11% of policy
weights differ from the start even at bf16 resolution; the reference is unchanged). The rule
measures termination and length only; a shorter answer is not a better answer, and nothing here
measures helpfulness (one sampled answer even switched language). One run with one seed: a
prototype run with the same settings gave +0.42 ± 0.05 and −33 ± 4 tokens, so the direction is
stable but the size varies from run to run (GPU sampling is not bit-reproducible), and it depends
on the learning rate, β and the number of iterations.

**Next stage.** [S4 · GRPO](s4_grpo.ipynb): drop the value model. The advantage of each answer
becomes its reward relative to a group of answers to the same prompt (RLOO's idea from Part B,
standardized), with the same clipped ratio loss and a KL term moved into the loss.